# Lab 3 — Train a real gesture classifier
Upload CSVs (collected from at least two different sessions), train a small network, and export model artifacts for Labs 4–7.
**Important:** split by collection session/person, not by randomly splitting highly correlated video frames.


In [ ]:
!pip -q install torch scikit-learn pandas onnx onnxruntime


In [ ]:
# Upload at least 2 session CSVs created by web app's Lab 2 Capture panel.
from google.colab import files
uploaded=files.upload()
import pandas as pd, numpy as np
raw=pd.concat([pd.read_csv(name) for name in uploaded if name.endswith('.csv')],ignore_index=True)
FEATURES=[f'f{i}' for i in range(42)]
CLASSES=['open_palm','fist','peace','thumbs_up']
assert all(c in raw for c in ['label','session']+FEATURES), 'Unexpected schema'
raw=raw[raw.label.isin(CLASSES)].dropna(subset=FEATURES)
print('Counts:',raw.label.value_counts().to_dict())
print('Sessions:',raw.session.unique())
raw.head()


In [ ]:
import torch, torch.nn as nn
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt, json
np.random.seed(7);torch.manual_seed(7)
if raw.session.nunique()<2: raise ValueError('Collect a second distinct session before training.')
X=raw[FEATURES].to_numpy(dtype=np.float32)
y=np.array([CLASSES.index(s) for s in raw.label],dtype=np.int64)
groups=raw.session.to_numpy()
sp=GroupShuffleSplit(n_splits=1,test_size=.25,random_state=7)
train_idx,test_idx=next(sp.split(X,y,groups))
print('TRAIN SESSIONS',set(groups[train_idx]),'TEST SESSIONS',set(groups[test_idx]))
assert len(set(groups[train_idx]) & set(groups[test_idx])) == 0
print('Train/test examples',len(train_idx),len(test_idx))
# If any class is absent from train, collect more sessions.
assert set(y[train_idx])==set(range(4)), 'Training split missing a gesture. Collect more sessions.'


In [ ]:
class GestureMLP(nn.Module):
  def __init__(self,hidden=64):
    super().__init__()
    self.layers=nn.Sequential(nn.Linear(42,hidden),nn.ReLU(),nn.Linear(hidden,32),nn.ReLU(),nn.Linear(32,4))
  def forward(self,x):return self.layers(x)
model=GestureMLP(hidden=64) # TODO: change hidden=64 to hidden=32 and compare
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)
loss_fn=nn.CrossEntropyLoss()
Xtr=torch.from_numpy(X[train_idx]);ytr=torch.from_numpy(y[train_idx]);Xte=torch.from_numpy(X[test_idx]);yte=torch.from_numpy(y[test_idx])
losses=[];accuracies=[]
for epoch in range(40):
  model.train();order=torch.randperm(len(Xtr));running=0
  for ix in order.split(64):
    logits=model(Xtr[ix]);loss=loss_fn(logits,ytr[ix]);optimizer.zero_grad();loss.backward();optimizer.step();running+=loss.item()*len(ix)
  model.eval()
  with torch.no_grad(): acc=(model(Xte).argmax(1)==yte).float().mean().item()
  losses.append(running/len(Xtr));accuracies.append(acc)
print('Held-out session accuracy:',accuracies[-1])
plt.plot(losses);plt.xlabel('Epoch');plt.ylabel('Training loss');plt.show()
plt.plot(accuracies);plt.xlabel('Epoch');plt.ylabel('Held-out session accuracy');plt.show()


In [ ]:
model.eval()
with torch.no_grad(): predictions=model(Xte).argmax(1).numpy()
print(classification_report(y[test_idx],predictions,labels=list(range(4)),target_names=CLASSES,zero_division=0))
print('Confusion matrix:\n',confusion_matrix(y[test_idx],predictions,labels=list(range(4))))
torch.save({'state_dict':model.state_dict(),'hidden':model.layers[0].out_features,'classes':CLASSES},'gesture_model.pth')
with open('classes.json','w') as f:json.dump(CLASSES,f)
# The model output is unnormalized logits; browser app applies softmax.
torch.onnx.export(model,torch.randn(1,42),'gesture_model.onnx',input_names=['features'],output_names=['logits'],opset_version=17,dynamic_axes={'features':{0:'batch'},'logits':{0:'batch'}})
import onnx
onnx.checker.check_model(onnx.load('gesture_model.onnx'))
from google.colab import files
for path in ['gesture_model.pth','gesture_model.onnx','classes.json']:files.download(path)
# TODO: Verify actual hidden size in checkpoint if you changed the architecture.
